In [2]:
sample = sorted(glob.glob("/content/depth/train/depth/*.png"))[0]
name = os.path.basename(sample)
d = np.array(Image.open(sample)).astype(np.float32) / 100.0
img = np.array(Image.open(f"/content/depth/train/images/{name}"))
print("sample:", name)
print("thermal image shape:", img.shape, " depth shape:", d.shape)
print(f"depth range: {d.min():.2f} m to {d.max():.2f} m, sky/far pixels (>=99.5 m): {(d >= 99.5).mean()*100:.1f}%")
!nvidia-smi --query-gpu=name --format=csv

sample: pole_cam_20261001_162542_00000.png
thermal image shape: (256, 320)  depth shape: (256, 320)
depth range: 9.16 m to 100.00 m, sky/far pixels (>=99.5 m): 10.0%
name
Tesla T4


In [4]:
import math, time, torch, torch.nn as nn, torch.nn.functional as F
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)
DEV = "cuda"
ROOT = "/content/depth"
MAX_D, SKY_D = 100.0, 99.5
SAVE_DIR = f"{D}/runs/depth"
os.makedirs(SAVE_DIR, exist_ok=True)

def load_split(split):
    names = sorted(os.listdir(f"{ROOT}/{split}/images"))
    imgs = np.stack([np.array(Image.open(f"{ROOT}/{split}/images/{n}")) for n in names])
    deps = np.stack([np.array(Image.open(f"{ROOT}/{split}/depth/{n}")) for n in names])
    return names, imgs, deps            # uint8 thermal, uint16 depth in cm

class DepthSet(Dataset):
    def __init__(self, split, augment=False):
        self.names, self.imgs, self.deps = load_split(split)
        self.augment = augment
    def __len__(self):
        return len(self.names)
    def __getitem__(self, i):
        x = torch.from_numpy(self.imgs[i]).float().div(255).sub(0.5).div(0.25)[None]
        y = torch.from_numpy(self.deps[i].astype(np.float32) / 100.0)[None]
        if self.augment and torch.rand(1) < 0.5:
            x, y = x.flip(-1), y.flip(-1)
        return x, y

train_ds, val_ds = DepthSet("train", augment=True), DepthSet("val")
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)
print("train", len(train_ds), "val", len(val_ds))

model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=1, classes=1).to(DEV)

def predict(m, x):
    return torch.exp(m(x)).clamp(0.1, MAX_D)   # network outputs log-depth

def silog_loss(pred, gt):
    valid = (gt > 0.1) & (gt < SKY_D)
    d = torch.log(pred[valid]) - torch.log(gt[valid])
    return torch.sqrt((d ** 2).mean() - 0.85 * d.mean() ** 2 + 1e-8) * 10

@torch.no_grad()
def depth_metrics(m, dl):
    m.eval()
    tot = {"rmse": 0, "absrel": 0, "d1": 0, "n": 0}
    for x, y in dl:
        x, y = x.to(DEV), y.to(DEV)
        p = predict(m, x)
        for pi, yi in zip(p, y):
            v = (yi > 0.1) & (yi < SKY_D)
            a, b = pi[v], yi[v]
            tot["rmse"] += torch.sqrt(((a - b) ** 2).mean()).item()
            tot["absrel"] += ((a - b).abs() / b).mean().item()
            tot["d1"] += (torch.maximum(a / b, b / a) < 1.25).float().mean().item()
            tot["n"] += 1
    return {k: tot[k] / tot["n"] for k in ("rmse", "absrel", "d1")}

EPOCHS = 40
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
scaler = torch.amp.GradScaler()
best = float("inf")
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train()
    run = 0.0
    for x, y in train_dl:
        x, y = x.to(DEV), y.to(DEV)
        opt.zero_grad()
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x)
        loss = silog_loss(torch.exp(out.float()).clamp(0.1, MAX_D), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        run += loss.item()
    mt = depth_metrics(model, val_dl)
    flag = ""
    if mt["rmse"] < best:
        best = mt["rmse"]
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_depth.pt")
        flag = "  <- best, saved"
    print(f"epoch {ep:2d}  loss {run/len(train_dl):.3f}  val RMSE {mt['rmse']:.2f} m  "
          f"AbsRel {mt['absrel']:.3f}  d1 {mt['d1']:.3f}  ({time.time()-t0:.0f}s){flag}")
print("best val RMSE:", round(best, 3), "m  -> saved to", f"{SAVE_DIR}/best_depth.pt")

train 1862 val 270


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_691/3583181558.py:80: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


epoch  1  loss 10.777  val RMSE 22.05 m  AbsRel 0.719  d1 0.012  (23s)  <- best, saved
epoch  2  loss 4.208  val RMSE 10.01 m  AbsRel 0.281  d1 0.212  (44s)  <- best, saved
epoch  3  loss 1.974  val RMSE 7.94 m  AbsRel 0.161  d1 0.699  (64s)  <- best, saved
epoch  4  loss 1.508  val RMSE 5.17 m  AbsRel 0.100  d1 0.907  (85s)  <- best, saved
epoch  5  loss 1.230  val RMSE 3.39 m  AbsRel 0.063  d1 0.972  (108s)  <- best, saved
epoch  6  loss 1.037  val RMSE 3.16 m  AbsRel 0.060  d1 0.978  (130s)  <- best, saved
epoch  7  loss 0.982  val RMSE 2.88 m  AbsRel 0.058  d1 0.980  (152s)  <- best, saved
epoch  8  loss 1.015  val RMSE 5.40 m  AbsRel 0.113  d1 0.883  (173s)
epoch  9  loss 1.390  val RMSE 5.95 m  AbsRel 0.098  d1 0.924  (194s)
epoch 10  loss 1.012  val RMSE 2.91 m  AbsRel 0.059  d1 0.978  (215s)
epoch 11  loss 1.014  val RMSE 2.88 m  AbsRel 0.056  d1 0.978  (236s)
epoch 12  loss 0.862  val RMSE 2.70 m  AbsRel 0.050  d1 0.980  (257s)  <- best, saved
epoch 13  loss 1.094  val RMSE 2.

In [5]:
model.load_state_dict(torch.load(f"{SAVE_DIR}/best_depth.pt", map_location=DEV))
model.eval()
test_names, test_imgs, test_deps = load_split("test")
train_names, _, train_deps = load_split("train")

def agent_of(n):
    return "pole_cam" if n.startswith("pole_cam") else n.split("_")[0]

# baseline: per-agent mean training depth map (ignores the image entirely)
mean_map = {}
for ag in ("pole_cam", "ugv", "uav"):
    idx = [i for i, n in enumerate(train_names) if agent_of(n) == ag]
    dd = train_deps[idx].astype(np.float32) / 100.0
    v = (dd > 0.1) & (dd < SKY_D)
    mean_map[ag] = np.where(v.sum(0) > 0, (dd * v).sum(0) / np.maximum(v.sum(0), 1), MAX_D)

@torch.no_grad()
def predict_np(img_u8):
    x = torch.from_numpy(img_u8).float().div(255).sub(0.5).div(0.25)[None, None].to(DEV)
    return predict(model, x)[0, 0].cpu().numpy()

def img_metrics(p, g):
    v = (g > 0.1) & (g < SKY_D)
    a, b = p[v], g[v]
    return np.sqrt(((a - b) ** 2).mean()), (np.abs(a - b) / b).mean(), (np.maximum(a / b, b / a) < 1.25).mean()

rows = {}
person_err = {"model": [], "baseline": []}
person_rel = {"model": [], "baseline": []}
H, W = 256, 320
for i, n in enumerate(test_names):
    ag = agent_of(n)
    g = test_deps[i].astype(np.float32) / 100.0
    p = predict_np(test_imgs[i])
    b = mean_map[ag]
    for tag, pred in (("model", p), ("baseline", b)):
        rows.setdefault((ag, tag), []).append(img_metrics(pred, g))
        rows.setdefault(("all", tag), []).append(img_metrics(pred, g))
    for line in open(f"{ROOT}/test/boxes/{n[:-4]}.txt"):
        c, x, y, bw, bh = map(float, line.split()[:5])
        if int(c) != 0:
            continue
        x0, x1 = int((x - bw / 2) * W), int(np.ceil((x + bw / 2) * W))
        y0, y1 = int((y - bh / 2) * H), int(np.ceil((y + bh / 2) * H))
        gb = g[y0:y1, x0:x1]
        if gb.size == 0:
            continue
        true_d = np.median(gb)
        for tag, pred in (("model", p), ("baseline", b)):
            est = np.median(pred[y0:y1, x0:x1])
            person_err[tag].append(abs(est - true_d))
            person_rel[tag].append(abs(est - true_d) / true_d)

lines = ["| Agent | Method | RMSE (m) | AbsRel | d1 |", "|---|---|---|---|---|"]
for ag in ("all", "pole_cam", "ugv", "uav"):
    for tag in ("model", "baseline"):
        m = np.array(rows[(ag, tag)]).mean(0)
        lines.append(f"| {ag} | {'U-Net (ours)' if tag == 'model' else 'mean-depth baseline'} | {m[0]:.2f} | {m[1]:.3f} | {m[2]:.3f} |")
lines += ["", "| Person distance | Method | Mean abs error (m) | Median abs error (m) | Mean rel. error |", "|---|---|---|---|---|"]
for tag in ("model", "baseline"):
    e, r = np.array(person_err[tag]), np.array(person_rel[tag])
    lines.append(f"| {len(e)} people | {'U-Net (ours)' if tag == 'model' else 'mean-depth baseline'} | {e.mean():.2f} | {np.median(e):.2f} | {r.mean():.3f} |")
report = "\n".join(lines)
print(report)
open(f"{SAVE_DIR}/depth_test_results.md", "w").write("# Deliverable 2: thermal depth, test set (sky excluded)\n\n" + report + "\n")
print("\nsaved to", f"{SAVE_DIR}/depth_test_results.md")

| Agent | Method | RMSE (m) | AbsRel | d1 |
|---|---|---|---|---|
| all | U-Net (ours) | 1.89 | 0.028 | 0.988 |
| all | mean-depth baseline | 4.15 | 0.086 | 0.886 |
| pole_cam | U-Net (ours) | 1.42 | 0.019 | 0.991 |
| pole_cam | mean-depth baseline | 0.22 | 0.001 | 1.000 |
| ugv | U-Net (ours) | 3.26 | 0.032 | 0.980 |
| ugv | mean-depth baseline | 9.07 | 0.110 | 0.868 |
| uav | U-Net (ours) | 1.00 | 0.034 | 0.992 |
| uav | mean-depth baseline | 3.17 | 0.147 | 0.790 |

| Person distance | Method | Mean abs error (m) | Median abs error (m) | Mean rel. error |
|---|---|---|---|---|
| 768 people | U-Net (ours) | 3.65 | 1.11 | 0.125 |
| 768 people | mean-depth baseline | 13.73 | 2.90 | 0.543 |

saved to /content/drive/MyDrive/thermal-mas-v2/runs/depth/depth_test_results.md


In [6]:
results = {"box median": [], "box 20th percentile": [], "hottest pixels": []}
for i, n in enumerate(test_names):
    g = test_deps[i].astype(np.float32) / 100.0
    t = test_imgs[i].astype(np.float32)
    p = predict_np(test_imgs[i])
    for line in open(f"{ROOT}/test/boxes/{n[:-4]}.txt"):
        c, x, y, bw, bh = map(float, line.split()[:5])
        if int(c) != 0:
            continue
        x0, x1 = int((x - bw / 2) * W), int(np.ceil((x + bw / 2) * W))
        y0, y1 = int((y - bh / 2) * H), int(np.ceil((y + bh / 2) * H))
        tb, gb, pb = t[y0:y1, x0:x1], g[y0:y1, x0:x1], p[y0:y1, x0:x1]
        if tb.size < 4:
            continue
        hot = tb >= np.percentile(tb, 75)          # hottest quarter of the box = the person
        truth = np.median(gb[hot])                 # true distance to the person's own pixels
        ests = {"box median": np.median(pb),
                "box 20th percentile": np.percentile(pb, 20),
                "hottest pixels": np.median(pb[hot])}
        for k, e in ests.items():
            results[k].append((abs(e - truth), abs(e - truth) / truth))

lines = ["| Person distance estimate | People | Mean abs error (m) | Median abs error (m) | Mean rel. error | Within 10% |",
         "|---|---|---|---|---|---|"]
for k, v in results.items():
    a = np.array(v)
    lines.append(f"| {k} | {len(a)} | {a[:,0].mean():.2f} | {np.median(a[:,0]):.2f} | {a[:,1].mean():.3f} | {(a[:,1] < 0.10).mean():.3f} |")
rep = "\n".join(lines)
print(rep)
open(f"{SAVE_DIR}/depth_test_results.md", "a").write("\n## Person distance estimators (truth = depth at hottest pixels in box)\n\n" + rep + "\n")
print("appended to depth_test_results.md")

| Person distance estimate | People | Mean abs error (m) | Median abs error (m) | Mean rel. error | Within 10% |
|---|---|---|---|---|---|
| box median | 768 | 3.59 | 1.54 | 0.156 | 0.643 |
| box 20th percentile | 768 | 1.56 | 0.86 | 0.064 | 0.785 |
| hottest pixels | 768 | 1.80 | 1.21 | 0.075 | 0.764 |
appended to depth_test_results.md
